#Model Training TF-IDF

## This notebook illustrates the training of the TF-IDF model by loading both resumes text and job description text

## 1. Load all resume text in an array

In [1]:
import os
from pathlib import Path

resume_texts = []

resume_folder = Path("data/resumes_txt")

for root, dirs, files in os.walk(str(resume_folder)):
    for file in files:
        if file.endswith(".txt"):
            path = os.path.join(root, file)

            with open(path, 'r', encoding='utf-8') as f:
                resume_texts.append(f.read())
print("Resumes loaded: ", len(resume_texts))


Resumes loaded:  2484


## 2. Load all job descriptions in an array

In [2]:
job_description_texts = []

job_description_folder = Path("data/job_descriptions")

for root, dirs, files in os.walk(str(job_description_folder)):
    for file in files:
        if file.endswith(".txt"):
            path = os.path.join(root, file)

            with open(path, 'r', encoding='utf-8') as f:
                job_description_texts.append(f.read())

print("Job descriptions loaded: ", len(job_description_texts))

Job descriptions loaded:  23


## 3. Combine everything for training (Resume data + job descriptions)
This way the model sees all vocabulary that is relevant for matching.
1. 2484 resumes give the vectorizer a solid vacabulary and a good IDF calculation.
2. 23 job descriptions include relevant terms for job descriptions

TF-IDF does not require a huge dataset like deep learning.
Balance TF-IDF  will naturally down-weight words that appear in almost all resumes like education, and highlight words that appear more selectively

In [3]:
# Training corpus
corpus = resume_texts + job_description_texts

## 4. Cleaning the text using the clean_text function in text_cleaner.py

In [4]:
from src.utils.text_cleaner import clean_text

#cleaning all the data
clean_corpus = [clean_text(doc) for doc in corpus]
print("Clean corpus is ready")

[nltk_data] Downloading package stopwords to C:\Users\PHILLIPPA
[nltk_data]     SANYAMAHWE\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\PHILLIPPA
[nltk_data]     SANYAMAHWE\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Clean corpus is ready


## 5. TF-IDF vectorization
### What happens here the model:
1. learns vocabulary
2. learns IDF scores
3. creates vectors

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),  # capture phrases like "machine learning"
    min_df=2,           # ignore very rare words
    max_df=0.9          #ignore extremely common words
)
tfid_matrix = vectorizer.fit_transform(clean_corpus)

In [7]:
print(tfid_matrix.shape)

(2507, 167959)


## 6. Saving the trained data

In [8]:
import pickle

with open("tfidf_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)

## 7. Testing the system in Notebook

In [9]:
job = clean_text("python developer machine learning")

In [10]:
job_vector = vectorizer.transform([job])

In [11]:
resume_vectors = vectorizer.transform(resume_texts)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(job_vector, resume_vectors)

print(scores)

[[0.         0.         0.         ... 0.06441202 0.02584629 0.00420995]]


In [13]:
import numpy as np

scores= scores.flatten()

top5_indices = np.argsort(scores)[::-1][:5]

print("Top 5 resume indices:", top5_indices)
print("Scores:", scores[top5_indices])

Top 5 resume indices: [ 733  516 1520 2272 1464]
Scores: [0.15024228 0.13046662 0.12487517 0.1184544  0.11163853]
